# SME-DataTrust BI — configuración
Ejecuta los notebooks del 01 al 06 en orden y dentro de la misma sesión de Colab. Si tu carpeta del proyecto está en otra ruta, modifica `PROJECT_DIR`.

In [ ]:
from pathlib import Path
import sys
candidates=[Path("/content/BI-PYMES-ML"),Path("/content/SME-DataTrust-BI"),Path.cwd()]
PROJECT_DIR=next((p for p in candidates if (p/"src").exists()),Path.cwd())
if not (PROJECT_DIR/"src").exists():
    raise FileNotFoundError("No se encontró src/. Ajusta PROJECT_DIR a la raíz del repositorio.")
sys.path.insert(0,str(PROJECT_DIR))
for d in ["data/raw","data/processed","data/experimental","results/figures","results/metrics"]:
    (PROJECT_DIR/d).mkdir(parents=True,exist_ok=True)
print("Proyecto:",PROJECT_DIR)

## 5. Local Outlier Factor y línea base estadística
LOF se ajusta sobre el conjunto disponible (`novelty=False`). Sus métricas en este notebook describen detección sobre ese conjunto, no generalización a datos futuros.

In [ ]:
import pandas as pd
from src.anomaly_detection import fit_local_outlier_factor,statistical_baseline
p=PROJECT_DIR/"data/processed/invoice_features.pkl"
if not p.exists(): raise FileNotFoundError("Ejecuta primero Notebook 03.")
invoice=pd.read_pickle(p)
model,lof,features=fit_local_outlier_factor(invoice,contamination=.03,n_neighbors=20)
baseline,_=statistical_baseline(invoice)
print("LOF señaladas:",int(lof.lof_anomaly.sum()))
print("Baseline señaladas:",int(baseline.baseline_anomaly.sum()))
display(lof.sort_values("lof_score",ascending=False).head(15))
lof.to_csv(PROJECT_DIR/"results/metrics/local_outlier_factor_scores.csv",index=False)
baseline.to_csv(PROJECT_DIR/"results/metrics/statistical_baseline_scores.csv",index=False)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7,4))
plt.bar(["LOF","Robust Z"],[lof.lof_anomaly.sum(),baseline.baseline_anomaly.sum()])
plt.ylabel("Facturas señaladas"); plt.title("Conteo de señales por método")
plt.tight_layout(); plt.savefig(PROJECT_DIR/"results/figures/model_flag_counts.png",dpi=160); plt.show()